In [30]:
import pandas as pd
df = pd.read_csv('D:/Downloads/merge.csv', low_memory=False)

### Stage label

In [31]:
df.value_counts('Stage')

Stage
Reconnaissance        34794
Lateral Movement      27445
Establish Foothold    27111
Data Exfiltration      7522
Cover up                362
Name: count, dtype: int64

In [32]:
list_features = [
    'bidirectional_stddev_ps',
    'bidirectional_mean_ps',
    'dst2src_duration_ms',
    'src2dst_duration_ms',
    'bidirectional_bytes',
    'bidirectional_packets',
    'bidirectional_duration_ms',
    'src2dst_packets',
    'dst2src_packets',
    'src2dst_bytes',
    'dst2src_bytes',
    'bidirectional_mean_piat_ms',
    'bidirectional_stddev_piat_ms',
    'bidirectional_max_piat_ms',
    'bidirectional_min_piat_ms',
    'src2dst_mean_piat_ms',
    'src2dst_stddev_piat_ms',
    'src2dst_max_piat_ms',
    'src2dst_min_piat_ms',
    'dst2src_mean_piat_ms',
    'dst2src_stddev_piat_ms',
    'dst2src_max_piat_ms',
    'dst2src_min_piat_ms',
    'bidirectional_fin_packets',
    'bidirectional_syn_packets',
    'bidirectional_rst_packets',
    'bidirectional_psh_packets',
    'bidirectional_ack_packets',
    'bidirectional_urg_packets',
    'bidirectional_cwr_packets',
    'bidirectional_ece_packets',
    'time',
    'locate',
    'Stage',
    'Activity'
]

df = df[list_features]

In [33]:
# Giả sử stage_mapping là từ điển mà bạn đã định nghĩa
action_mapping = {
    'Maintain Access': 0,
    'Encrypted Channel: Symmetric Cryptography': 1,     
    'Data Transfer Size Limits': 2,
    'Remote System Discovery': 3,
    'Exfiltration over C2 channel': 4,
    'Remove Traces': 5,
    'Unsecured Credentials': 6,
    'Active Scanning: Scanning IP Blocks': 7,
    'Active Scanning: Vulnerability Scanning': 8,
    'Bruteforce: Password Guessing': 9
    
}
stage_mapping = {
    'Reconnaissance': 0,     
    'Establish Foothold': 1,
    'Lateral Movement': 2,
    'Data Exfiltration': 3,
    'Cover up': 4
}

In [34]:
action_test = df['Activity'].map(action_mapping)
stage_test = df['Stage'].map(stage_mapping)

In [35]:
df.drop(columns=['Activity', 'Stage'], inplace=True)

### Stage prediction

In [39]:
import joblib

stage_model = joblib.load("D:/code/python/code/ScoreFactor/main/model/stage_model.pkl")

In [40]:
stage_pred = stage_model.predict(df)

In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Calculate accuracy
accuracy = accuracy_score(stage_test, stage_pred)
print(accuracy)


conf_matrix = confusion_matrix(stage_test, stage_pred)
print(conf_matrix)

print(classification_report(stage_test, stage_pred))

0.9982516403727091
[[34768     2    24     0     0]
 [   10 27054     0    44     3]
 [   14     0 27424     7     0]
 [    1     5     0  7461    55]
 [    0     0     0     5   357]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     34794
           1       1.00      1.00      1.00     27111
           2       1.00      1.00      1.00     27445
           3       0.99      0.99      0.99      7522
           4       0.86      0.99      0.92       362

    accuracy                           1.00     97234
   macro avg       0.97      0.99      0.98     97234
weighted avg       1.00      1.00      1.00     97234


In [43]:
df['stage_label'] = stage_pred

In [44]:
df.to_csv('D:/Downloads/merge_label.csv', index=False)

### Action prediction

In [57]:
import joblib

action_model = joblib.load("D:/code/python/code/ScoreFactor/main/model/action_model.pkl")

In [58]:
df = pd.read_csv('D:/Downloads/merge_label.csv', low_memory=False)

In [59]:
# Danh sách các đặc trưng
list_features = [
    'bidirectional_stddev_ps',
    'bidirectional_mean_ps',
    'dst2src_duration_ms',
    'src2dst_duration_ms',
    'bidirectional_bytes',
    'bidirectional_packets',
    'bidirectional_duration_ms',
    'src2dst_packets',
    'dst2src_packets',
    'src2dst_bytes',
    'dst2src_bytes',
    'bidirectional_mean_piat_ms',
    'bidirectional_stddev_piat_ms',
    'bidirectional_max_piat_ms',
    'bidirectional_min_piat_ms',
    'src2dst_mean_piat_ms',
    'src2dst_stddev_piat_ms',
    'src2dst_max_piat_ms',
    'src2dst_min_piat_ms',
    'dst2src_mean_piat_ms',
    'dst2src_stddev_piat_ms',
    'dst2src_max_piat_ms',
    'dst2src_min_piat_ms',
    'bidirectional_fin_packets',
    'bidirectional_syn_packets',
    'bidirectional_rst_packets',
    'bidirectional_psh_packets',
    'bidirectional_ack_packets',
    'bidirectional_urg_packets',
    'bidirectional_cwr_packets',
    'bidirectional_ece_packets',
    'locate',
    'time',
    'stage_label',
]

# Lọc DataFrame theo list_features
df = df[list_features]

In [60]:
action_pred = action_model.predict(df)


In [61]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Calculate accuracy
accuracy = accuracy_score(action_test, action_pred)
print(accuracy)


conf_matrix = confusion_matrix(action_test, action_pred)
print(conf_matrix)

print(classification_report(action_test, action_pred))

0.9952999979431063
[[27123     2     0     0     0     0     0     1     0     2]
 [    0 27054    13     0    31     3     0     1     0     9]
 [    0     4  6983     0     0     0     0     0     0     1]
 [    0     0     1  1001     0     0     0     0     0     0]
 [    0     0     0     0   446     0     0     0     0     0]
 [    0     0     0     0     0   357     5     0     0     0]
 [    0     1     0     0     0    55    32     0     0     0]
 [    5     4     0     3     0     0     0 11420     8    10]
 [    0     0     0     0     0     0     0     0     5     0]
 [   16     2     0     0     0     0     0   263    17 22356]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     27128
           1       1.00      1.00      1.00     27111
           2       1.00      1.00      1.00      6988
           3       1.00      1.00      1.00      1002
           4       0.94      1.00      0.97       446
           5       0.86  

In [62]:
df['action_label'] = action_pred

In [63]:
df.to_csv('D:/Downloads/merge_label.csv', index=False)

### Who label

In [64]:
df = pd.read_csv('D:/Downloads/merge_label.csv', low_memory=False)

In [67]:
# Hàm gán nhãn 'who' với kiểm tra thêm Activity
def assign_who(row):
    # Điều kiện cho nhãn AA
    if (row['locate'] in [0, 4]) and (row['stage_label'] == 0) and (row['action_label'] in [7, 8, 9]):
        return 'AA'
    # Điều kiện cho nhãn APT
    elif (row['locate'] in [0, 3, 5]) and (row['stage_label'] in [0, 1, 2, 3, 4]) and (row['action_label'] in [0, 1, 2, 3, 4, 5, 6]):
        return 'APT'
    # Trường hợp không thỏa mãn điều kiện nào
    else:
        return 'Unknown'

In [68]:
# Áp dụng hàm assign_who cho từng hàng
df['who'] = df.apply(assign_who, axis=1)

In [70]:
# Kiểm tra kết quả
print("Số lượng mỗi nhãn trong cột 'who':")
print(df['who'].value_counts())
print("\nXem trước 5 dòng đầu của DataFrame:")
print(df[['locate', 'stage_label', 'who']].head())

Số lượng mỗi nhãn trong cột 'who':
who
APT        63137
AA         34083
Unknown       14
Name: count, dtype: int64

Xem trước 5 dòng đầu của DataFrame:
   locate  stage_label  who
0       3            0  APT
1       3            2  APT
2       3            0  APT
3       3            2  APT
4       3            0  APT


In [71]:
df.to_csv('D:/Downloads/merge_label.csv', index=False)

### defend label

In [72]:
df = pd.read_csv('D:/Downloads/merge_label.csv', low_memory=False)


In [73]:
# Đếm số dòng cho mỗi ngày
daily_counts = df.groupby('time').size().reset_index(name='Count')

# In kết quả
print("Số lượng dòng dữ liệu cho mỗi ngày:")
print(daily_counts)

Số lượng dòng dữ liệu cho mỗi ngày:
    time  Count
0     11     28
1     24    131
2     25    291
3     31    162
4     32     45
5     33     62
6     34     38
7     35     32
8     41     18
9     42     20
10    43     24
11    44     20
12    45     18
13    51    969
14    52   3561
15    53   5490
16    54   5835
17    55  11659
18    56  10614
19    61  39874
20    62  12169
21    63   3366
22    64   1420
23    66   1388


In [74]:
# Thêm cột 'defend' với giá trị ban đầu là 'detecting'
daily_counts['defend'] = 'detecting'

In [75]:
# Hàm tính phần trăm thay đổi và gán nhãn
def assign_defend_label(row, prev_count, prev_label):
    if prev_count is None:  # Ngày đầu tiên
        return 'detecting'
    
    current_count = row['Count']
    percent_change = (current_count - prev_count) / prev_count * 100
    
    if percent_change >= 20:
        return 'detecting'
    if percent_change <= -20:
        return 'restricting'
    else:
        return prev_label

In [76]:
# Gán nhãn 'defend' cho từng ngày
prev_count = None
prev_label = 'detecting'

for i, row in daily_counts.iterrows():
    daily_counts.loc[i, 'defend'] = assign_defend_label(row, prev_count, prev_label)
    prev_count = row['Count']
    prev_label = daily_counts.loc[i, 'defend']

In [77]:
# Gộp nhãn 'defend' vào DataFrame gốc
df['defend'] = df['time'].map(daily_counts.set_index('time')['defend'])

In [78]:
# In kết quả số dòng mỗi ngày và nhãn 'defend'
print("Số lượng dòng dữ liệu và nhãn defend cho mỗi ngày:")
print(daily_counts)

Số lượng dòng dữ liệu và nhãn defend cho mỗi ngày:
    time  Count       defend
0     11     28    detecting
1     24    131    detecting
2     25    291    detecting
3     31    162  restricting
4     32     45  restricting
5     33     62    detecting
6     34     38  restricting
7     35     32  restricting
8     41     18  restricting
9     42     20  restricting
10    43     24    detecting
11    44     20    detecting
12    45     18    detecting
13    51    969    detecting
14    52   3561    detecting
15    53   5490    detecting
16    54   5835    detecting
17    55  11659    detecting
18    56  10614    detecting
19    61  39874    detecting
20    62  12169  restricting
21    63   3366  restricting
22    64   1420  restricting
23    66   1388  restricting


In [79]:
df.to_csv('D:/Downloads/merge_label.csv', index=False)
